In [1]:
%pip install networkx

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# CELL 1 — 5-NN Geographic Graph Construction
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

MERGED_PATH = ROOT / "data/processed/cpcb_era5_2021_2025.csv"

K = 5
MAX_DIST_KM = 26.0
EARTH_KM = 6371.0

df = pd.read_csv(
    MERGED_PATH,
    usecols=["location_id","station_name","latitude","longitude"]
)

stations = (
    df.drop_duplicates("location_id")
      .sort_values("location_id")
      .reset_index(drop=True)
)

lat = np.radians(stations.latitude.to_numpy())
lon = np.radians(stations.longitude.to_numpy())

dlat = lat[:, None] - lat[None, :]
dlon = lon[:, None] - lon[None, :]

a = (
    np.sin(dlat/2)**2
    + np.cos(lat[:,None]) * np.cos(lat[None,:])
    * np.sin(dlon/2)**2
)

dist = EARTH_KM * 2 * np.arcsin(np.sqrt(a))

edges = []

for i in range(len(stations)):
    nearest = np.argsort(dist[i])[1:K+1]

    for j in nearest:
        if dist[i,j] <= MAX_DIST_KM:
            edges.append({
                "source": stations.location_id.iloc[i],
                "target": stations.location_id.iloc[j],
                "distance_km": dist[i,j]
            })

edges = pd.DataFrame(edges)

print("="*60)
print("INITIAL GRAPH")
print("="*60)
print("Stations:", len(stations))
print("Directed edges:", len(edges))

INITIAL GRAPH
Stations: 37
Directed edges: 167


In [3]:
# ============================================================
# CELL 2 — Connectivity Repair
# ============================================================

import networkx as nx

def components(edge_df):
    G = nx.Graph()
    G.add_nodes_from(stations.location_id)
    G.add_edges_from(
        edge_df[["source","target"]]
        .itertuples(index=False, name=None)
    )
    return list(nx.connected_components(G))

comps = components(edges)

print("Initial components:", len(comps))

while len(comps) > 1:
    best = None

    for c1 in comps:
        for c2 in comps:
            if c1 >= c2:
                continue

            for s1 in c1:
                i = stations.index[
                    stations.location_id == s1
                ][0]

                for s2 in c2:
                    j = stations.index[
                        stations.location_id == s2
                    ][0]

                    if best is None or dist[i,j] < best[0]:
                        best = (dist[i,j], s1, s2)

    d, s1, s2 = best

    edges = pd.concat([
        edges,
        pd.DataFrame([
            {"source":s1, "target":s2, "distance_km":d},
            {"source":s2, "target":s1, "distance_km":d}
        ])
    ], ignore_index=True)

    comps = components(edges)

print("="*60)
print("GRAPH CONSTRUCTION")
print("="*60)
print(f"Stations          : {len(stations)}")
print(f"Directed edges    : {len(edges)}")
print(f"K                 : {K}")
print(f"Distance cutoff   : {MAX_DIST_KM} km")
print(f"Max edge distance : {edges.distance_km.max():.2f} km")
print(f"Components        : {len(comps)}")

Initial components: 2
GRAPH CONSTRUCTION
Stations          : 37
Directed edges    : 169
K                 : 5
Distance cutoff   : 26.0 km
Max edge distance : 29.20 km
Components        : 1


In [4]:
# ============================================================
# CELL 3 — Graph Validation
# ============================================================

out_deg = edges.groupby("source").size()

stats = stations[
    ["location_id","station_name"]
].copy()

stats["out_degree"] = (
    stats.location_id.map(out_deg)
    .fillna(0)
    .astype(int)
)

print("="*60)
print("GRAPH VALIDATION")
print("="*60)

print(f"Nodes       : {len(stations)}")
print(f"Edges       : {len(edges)}")
print(f"Components  : {len(comps)}")
print(f"Min degree  : {stats.out_degree.min()}")
print(f"Max degree  : {stats.out_degree.max()}")
print(f"Mean degree : {stats.out_degree.mean():.2f}")

print("\nConnectivity:")
for i, c in enumerate(
    sorted(comps, key=len, reverse=True), 1
):
    print(f"Component {i}: {len(c)} stations")

print("\nLongest connections:")
print(
    edges.nlargest(10, "distance_km")
         .to_string(index=False)
)

GRAPH VALIDATION
Nodes       : 37
Edges       : 169
Components  : 1
Min degree  : 1
Max degree  : 6
Mean degree : 4.57

Connectivity:
Component 1: 37 stations

Longest connections:
source target  distance_km
  ST27   ST03    29.202982
  ST03   ST27    29.202982
  ST03   ST12    25.382132
  ST12   ST03    25.382132
  ST34   ST30    25.347787
  ST28   ST20    23.364890
  ST29   ST19    23.296422
  ST28   ST15    23.084067
  ST30   ST31    22.788420
  ST30   ST06    22.708587


In [5]:
# ============================================================
# CELL 4 — Save Graph Artifacts
# ============================================================

GRAPH_DIR = ROOT / "graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

stations.to_csv(
    GRAPH_DIR / "station_metadata_final.csv",
    index=False
)

edges.to_csv(
    GRAPH_DIR / "edge_index_final.csv",
    index=False
)

idx = {
    sid: i
    for i, sid in enumerate(stations.location_id)
}

edge_idx = np.array([
    [idx[s] for s in edges.source],
    [idx[t] for t in edges.target]
])

np.save(
    GRAPH_DIR / "graph_edge_index.npy",
    edge_idx
)

print("="*60)
print("GRAPH SAVED")
print("="*60)
print(f"Nodes : {len(stations)}")
print(f"Edges : {len(edges)}")
print("✓ graph/station_metadata_final.csv")
print("✓ graph/edge_index_final.csv")
print("✓ graph/graph_edge_index.npy")

GRAPH SAVED
Nodes : 37
Edges : 169
✓ graph/station_metadata_final.csv
✓ graph/edge_index_final.csv
✓ graph/graph_edge_index.npy


In [6]:
# ============================================================
# CELL 5 — Final Graph Audit
# ============================================================

print("="*60)
print("FINAL GRAPH AUDIT")
print("="*60)

print("Stations:", len(stations))
print("Edges:", len(edges))
print("Edge index shape:", edge_idx.shape)
print("Connected:", len(comps) == 1)
print("Min degree:", stats.out_degree.min())
print("Max degree:", stats.out_degree.max())
print("Duplicate edges:",
      edges.duplicated(["source","target"]).sum())

FINAL GRAPH AUDIT
Stations: 37
Edges: 169
Edge index shape: (2, 169)
Connected: True
Min degree: 1
Max degree: 6
Duplicate edges: 0
